# KDD Process Volcano Data Analysis

In [34]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

## Data Cleaning and Preprocessing

### A FAIRE:

Data Collection: Create a notebook, upload the data and store it in a structure that allows
data manipulation. Explore the data using the data summarization methods provided by Python to
fully understand it. You can use the statements we have learned in the course. At this stage, you should
at least show the number of columns and data types in each column, the number of rows and columns
in the dataset, the number of missing data points in each column, and the ranking between variables.
You can use the markdown boxes to describe the dataset at your disposal and some visual components
to help you understand your data. This last result can be used in the next step.

In [35]:
def load_data(filepath="volcano-events.tsv"):
    try:
        df = pd.read_csv(filepath, sep='\t')
    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found.")
        return pd.DataFrame()

    df.rename(columns={
        'Damage ($Mil)': 'Damage_Millions',
        'Total Damage ($Mil)': 'Total_Damage_Millions',
        'Elevation (m)': 'Elevation',
        'Total Deaths': 'Total_Deaths',
        'Total Injuries': 'Total_Injuries'
    }, inplace=True)

    df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
    df.dropna(subset=['Year'], inplace=True)
    df['VEI'] = pd.to_numeric(df['VEI'], errors='coerce')
    df['Deaths'] = pd.to_numeric(df['Deaths'], errors='coerce').fillna(0)
    df['Damage_Millions'] = pd.to_numeric(df['Damage_Millions'], errors='coerce').fillna(0)
    df.dropna(subset=['Latitude', 'Longitude', 'Country'], inplace=True)
    return df

## Spatial Analysis

In [36]:
def get_map_figure(df):
    df_map = df.copy()
    df_map['VEI_Size'] = df_map['VEI'].fillna(0.5)

    fig = px.scatter_geo(
        df_map,
        lat="Latitude",
        lon="Longitude",
        color="Type",
        size="VEI_Size",
        hover_name="Name",
        hover_data={"Country": True, "Year": True, "Deaths": True, "VEI_Size": False},
        title="Global Volcano Distribution",
        projection="natural earth",
        size_max=15,
        template="plotly_dark",
        color_discrete_sequence=px.colors.qualitative.Bold
    )
    fig.update_layout(
        margin={"r":0,"t":50,"l":0,"b":0},
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Temporal Analysis

In [37]:
def get_frequency_figure(df):
    fig = px.histogram(
        df, 
        x="Year", 
        title="Eruption Frequency",
        nbins=100,
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722')
    fig.update_layout(
        xaxis_title="Year", 
        yaxis_title="Count",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Impact Analysis

In [38]:
def get_impact_figure(df):
    top_deadly = df.nlargest(10, 'Deaths').sort_values('Deaths', ascending=True)
    fig = px.bar(
        top_deadly,
        x="Deaths",
        y="Name",
        orientation='h',
        text="Deaths",
        title="Top 10 Deadliest Eruptions",
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722', textposition='outside')
    fig.update_layout(
        xaxis_title="Deaths", 
        yaxis_title="",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Correlation Analysis

In [39]:
def get_correlation_figure(df):
    damage_df = df[df['Damage_Millions'] > 0].copy()
    if damage_df.empty:
         return px.scatter(title="No Data")

    fig = px.scatter(
        damage_df,
        x="VEI",
        y="Damage_Millions",
        size="Deaths",
        hover_name="Name",
        log_y=True,
        title="VEI vs. Impact",
        template="plotly_dark"
    )
    fig.update_traces(marker=dict(color='#ff5722', opacity=0.7))
    fig.update_layout(
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## UI for Dashboard

In [40]:
# Initialize App
app = Dash(__name__)

# Load Data
df = load_data()

# Styles
SIDEBAR_STYLE = {
    "position": "fixed",
    "top": 0,
    "left": 0,
    "bottom": 0,
    "width": "16rem",
    "padding": "2rem 1rem",
    "background-color": "#111111",
    "color": "white"
}

CONTENT_STYLE = {
    "margin-left": "18rem",
    "margin-right": "2rem",
    "padding": "2rem 1rem",
    "background-color": "#000000",
    "min-height": "100vh",
    "color": "white"
}

CARD_STYLE = {
    "background-color": "#1e1e1e",
    "padding": "20px",
    "border-radius": "10px",
    "margin-bottom": "20px",
    "box-shadow": "0 4px 6px rgba(0,0,0,0.3)"
}

# Layout
app.layout = html.Div([
    # Sidebar
    html.Div([
        html.H2("Volcano Insights", style={'font-size': '20px', 'margin-bottom': '20px', 'color': '#ff5722'}),
        html.Hr(style={'border-color': '#333'}),
        html.P("Filters", style={'color': '#888'}),
        
        html.Label("Year Range", style={'margin-top': '20px'}),
        dcc.RangeSlider(
            id='year-slider',
            min=df['Year'].min(),
            max=df['Year'].max(),
            value=[df['Year'].min(), df['Year'].max()],
            marks={str(year): str(year) for year in range(int(df['Year'].min()), int(df['Year'].max()), 1000)},
            tooltip={"placement": "bottom", "always_visible": True},
            className="dark-slider"
        ),
        
        html.Label("Country", style={'margin-top': '20px'}),
        dcc.Dropdown(
            id='country-dropdown',
            options=[{'label': c, 'value': c} for c in sorted(df['Country'].unique())],
            placeholder="All Countries",
            style={'color': 'black'} # Dropdown text needs to be black to be visible on white bg of default dropdown
        )
    ], style=SIDEBAR_STYLE),

    # Main Content
    html.Div([
        html.H1("Volcano Insights Dashboard", style={'margin-bottom': '5px'}),
        html.P("Analyzing Significant Volcanic Eruptions", style={'color': '#888', 'margin-bottom': '30px'}),

        # KPI Row
        html.Div([
            html.Div([
                html.H4("Total Eruptions", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-eruptions', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Deaths", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-deaths', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Damage ($M)", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-damage', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex', 'justify-content': 'space-between', 'margin-bottom': '20px'}),

        # Charts Row 1
        html.Div([
            html.Div([dcc.Graph(id='map-graph')], style={**CARD_STYLE, 'flex': '2', 'margin-right': '20px'}),
            html.Div([dcc.Graph(id='time-graph')], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex', 'margin-bottom': '20px'}),

        # Charts Row 2
        html.Div([
            html.Div([dcc.Graph(id='impact-graph')], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([dcc.Graph(id='corr-graph')], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex'})

    ], style=CONTENT_STYLE)
])

# Callbacks
@app.callback(
    [Output('map-graph', 'figure'),
     Output('time-graph', 'figure'),
     Output('impact-graph', 'figure'),
     Output('corr-graph', 'figure'),
     Output('kpi-eruptions', 'children'),
     Output('kpi-deaths', 'children'),
     Output('kpi-damage', 'children')],
    [Input('country-dropdown', 'value'),
     Input('year-slider', 'value')]
)
def update_dashboard(selected_country, year_range):
    # Filter Data
    dff = df.copy()
    if selected_country:
        dff = dff[dff['Country'] == selected_country]
    
    if year_range:
        dff = dff[(dff['Year'] >= year_range[0]) & (dff['Year'] <= year_range[1])]

    if dff.empty:
        dff = df # Fallback if empty

    # KPIs
    total_eruptions = len(dff)
    total_deaths = f"{int(dff['Deaths'].sum()):,}"
    total_damage = f"${dff['Damage_Millions'].sum():,.0f}"

    # Figures
    fig1 = get_map_figure(dff)
    fig2 = get_frequency_figure(dff)
    fig3 = get_impact_figure(dff)
    fig4 = get_correlation_figure(dff)

    return fig1, fig2, fig3, fig4, total_eruptions, total_deaths, total_damage

if __name__ == '__main__':
    print("Launching Dashboard...")
    print("Dashboard launched at: http://127.0.0.1:7860")
    app.run(host='127.0.0.1', port=7860, debug=True)

Launching Dashboard...
Dashboard launched at: http://127.0.0.1:7860
